# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Model
I will use a linear regression as a simple first pass, then Random Forest Regressor to see if it improves on nonlinear feature interactions. Both predict 'refresh_score' directly since it's a continuous target.

Among the pages with a valid trend signal(i.e. excluding the ~3,388 rows with no trend baseline), a page is labeled priority(is_priority = 1) if its 'refresh_score, falls in the top 10% of that subset; otherwise 0. This label exists only to give 'precision@k' a yes/no ground truth to check predictions against. Note that it does not replace 'refresh_score' as the regression target.

In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\hamto\OneDrive\Desktop\Flyrank-Machine-Learning-Internship\data\raw\content_refresh_anonymized.csv')
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
df['trend_pct_flipped'] = -df['trend_pct']
lower = df['trend_pct_flipped'].quantile(0.01)
upper = df['trend_pct_flipped'].quantile(0.99)

df['trend_pct_dampened'] = df['trend_pct_flipped'].clip(lower, upper)

In [3]:
df['refresh_score'] = df['trend_pct_dampened'] * df['search_volume'] * df['cpc']

In [4]:
df = df[df['trend_pct'].notna()]
threshold = df['refresh_score'].quantile(0.90)

df['is_priority'] = (df['refresh_score'] >= threshold).astype(int)
df['is_priority'].value_counts()

is_priority
0    24083
1     2529
Name: count, dtype: int64

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I'll use a grouped train/test split on client_id (via GroupShuffleSplit) rather than a random split, because pages from the same client share hidden site-level characteristics — a random split would let the model partly memorize client identity instead of learning generalizable patterns.

In [5]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

train_df = train_df[train_df['search_volume'].notna() & train_df['cpc'].notna()]
test_df = test_df[test_df['search_volume'].notna() & test_df['cpc'].notna()]

In [6]:
overlap = set(train_df['client_id']) & set(test_df['client_id'])
print(len(overlap))  # should be 0

0


In [7]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct', 'trend_pct_flipped',
       'trend_pct_dampened', 'refresh_score', 'is_priority'],
      dtype='str')

In [8]:
drop_cols = [
    'client_id', 'content_id', 'is_priority', 'trend_pct', 'trend_pct_flipped',
    'trend_pct_dampened', 'refresh_score', 'cpc', 'search_volume',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'trend_direction',
    'provider_used', 'model_used', 'age_tier', 'impression_tier', 'position_tier',
    'freshness_tier', 'char_count_tier', 'word_count_tier',
    ]  # your full list above
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['refresh_score']

X_test = test_df.drop(columns=drop_cols)
y_test = test_df['refresh_score']

X_train.dtypes

competition               float64
competition_level             str
content_type                  str
main_intent                   str
word_count                float64
char_count                float64
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
content_age_days            int64
age_tier_order              int64
days_since_last_update      int64
ctr                       float64
avg_position              float64
engagement_rate           float64
scroll_rate               float64
ai_traffic_pct            float64
dtype: object

In [9]:
train_df.groupby('content_type')['word_count'].apply(lambda x: x.isna().mean())

content_type
comparison article    0.000000
keyword article       0.332748
Name: word_count, dtype: float64

In [10]:
for col in ['word_count', 'char_count']:
    X_train[f'{col}_missing'] = X_train[col].isna().astype(int)
    X_test[f'{col}_missing'] = X_test[col].isna().astype(int)

    median_val = X_train[col].median()  # compute from TRAIN only
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)  # apply train's median to test too

In [11]:
# numeric — fill with train median
for col in ['competition', 'scroll_rate']:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

# categorical — explicit "missing" category, not a guess
for col in ['competition_level', 'main_intent']:
    X_train[col] = X_train[col].fillna('missing')
    X_test[col] = X_test[col].fillna('missing')

In [12]:
X_train.isna().sum().sum()
X_test.isna().sum().sum()

np.int64(0)

In [13]:
X_train_encoded = pd.get_dummies(X_train)
X_test_encoded = pd.get_dummies(X_test)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

print(X_train_encoded.shape)
print(X_test_encoded.shape)

(20635, 34)
(4652, 34)


In [14]:
y_train.isna().sum()

np.int64(0)

In [15]:
train_df[['trend_pct_dampened', 'search_volume', 'cpc']].isna().sum()

trend_pct_dampened    0
search_volume         0
cpc                   0
dtype: int64

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

lr = LinearRegression()
lr.fit(X_train_encoded, y_train)
lr_preds = lr.predict(X_test_encoded)

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_encoded, y_train)
rf_preds = rf.predict(X_test_encoded)

In [17]:
def precision_at_k(preds, actual, k):
    results = pd.DataFrame({'pred': preds, 'actual': actual})
    results_sorted = results.sort_values('pred', ascending=False)
    return (results_sorted.head(k)['actual'] == 1).mean()

In [18]:
precision_at_k(lr_preds, test_df['is_priority'].values, 20)

np.float64(0.2)

In [19]:
precision_at_k(rf_preds, test_df['is_priority'].values, 20)

np.float64(0.45)

In [20]:
test_df['is_priority'].mean()

np.float64(0.11349957007738606)

In [21]:
import numpy as np

# value + rank components — computed on test_df only
value = test_df['cpc'] * test_df['search_volume']
declining_rank = test_df['trend_pct_dampened'].abs().rank(pct=True)
staleness_rank = test_df['days_since_last_update'].rank(pct=True)

# masks
near_miss_mask = test_df['avg_position'].between(8, 20)
decline_threshold = test_df.loc[near_miss_mask, 'trend_pct_dampened'].quantile(0.75)
stale_threshold = 200

declining_only_mask = near_miss_mask & (test_df['trend_pct_dampened'] >= decline_threshold) & (test_df['days_since_last_update'] < stale_threshold)
stale_only_mask = near_miss_mask & (test_df['days_since_last_update'] >= stale_threshold) & (test_df['trend_pct_dampened'] < decline_threshold)
stale_declining_mask = near_miss_mask & (test_df['trend_pct_dampened'] >= decline_threshold) & (test_df['days_since_last_update'] >= stale_threshold)

# scores
score_declining = value * declining_rank
score_stale = value * staleness_rank
score_stale_declining = value * (declining_rank + staleness_rank)

conditions = [stale_declining_mask, declining_only_mask, stale_only_mask]
score_choices = [score_stale_declining, score_declining, score_stale]

test_df['baseline_score'] = np.select(conditions, score_choices, default=0)

In [22]:
precision_at_k(test_df['baseline_score'].values, test_df['is_priority'].values, 20)

np.float64(1.0)

## Comparison Talble
| Method | precision@20 | Note |
|---|---|---|
| Base rate (random) | 0.113 | — |
| Baseline rule | 1.0 | Invalid comparison — baseline score and `is_priority` share the same core ingredients (cpc × search_volume × decline), so this isn't a fair test |
| Linear Regression | 0.20 | ~1.8x base rate |
| Random Forest | 0.45 | ~4x base rate |

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [23]:
results_rf = pd.DataFrame({
    'content_id': test_df['content_id'].values,
    'pred': rf_preds,
    'actual': test_df['is_priority'].values,
    'avg_position': test_df['avg_position'].values,
    'cpc': test_df['cpc'].values,
    'search_volume': test_df['search_volume'].values,
    'days_since_last_update': test_df['days_since_last_update'].values,
})

top20_rf = results_rf.sort_values('pred', ascending=False).head(20)
wrong_picks = top20_rf[top20_rf['actual'] == 0]
wrong_picks

,content_id,pred,actual,avg_position,cpc,search_volume,days_since_last_update
1810,content_dd9ab01faf99,675665.990320,0,6.0,0.04,10.0,8
3834,content_0ac51e0dae75,286551.476000,0,6.8,0.40,1000.0,25
739,content_78afcf3ee0f4,255921.747100,0,8.9,0.09,10.0,25
4599,content_7fe7914ad0e0,243686.992300,0,9.6,0.00,20.0,7
100,content_0d78cba0b985,216979.647700,0,7.4,0.00,10.0,20
2239,content_ccc3299baa8a,206889.706000,0,20.4,0.15,880.0,13
2213,content_b32c56556a77,195835.576300,0,13.5,1.33,30.0,20
1269,content_1c19b0d3f7d0,192690.939330,0,5.8,2.26,170.0,20
498,content_eb7015b56be9,188955.845600,0,9.1,0.14,2900.0,13
998,content_b9e11a38fd49,180661.069284,0,7.2,2.69,10.0,25


## ERROR ANALYSIS
The model under-predicts the influence of commercial value because cpc and search_volume — direct multiplicative components of the label — were excluded as leakage. As a result, the model over-indexes on decline-adjacent signals and misranks low-value, recently-updated pages as high priority. A production version would need a value proxy that isn't leakage — e.g., competition_level, content_type, or historical value bucketed at a coarser grain than the exact cpc used in the label.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.